# 03 - Global VAR (GVAR) and Historical Decomposition

This notebook covers two advanced topics:

1. **GVAR (Global VAR)**: a multi-country framework that models international interdependencies through trade-weighted foreign variables
2. **Historical Decomposition**: decomposing observed time series into contributions from each structural shock

## Topics covered

- The GVAR framework (Pesaran, Schuermann & Weiner, 2004)
- Trade weight matrices and foreign variable construction
- Country-specific VARX models and the global solution
- Generalized IRFs (GIRF) for cross-country spillovers
- Historical decomposition via SVAR
- Spillover network analysis
- Exercises: analyzing global shock transmission

---

## Part I: The GVAR Model

The standard VAR treats the economy as a single unit. The GVAR extends this to **multiple interconnected countries**, each modeled by a VARX:

$$y_{it} = a_i + \Phi_i y_{i,t-1} + \Lambda_{i0} y^*_{it} + \Lambda_{i1} y^*_{i,t-1} + u_{it}$$

where $y^*_{it} = \sum_{j \neq i} w_{ij} y_{jt}$ are **foreign variables** constructed using bilateral trade weights $w_{ij}$.

The key innovation: by linking country models through trade weights, the GVAR captures **cross-border spillovers** while keeping each country model small and tractable.

**Key references:**
- Pesaran, M.H., Schuermann, T. & Weiner, S.M. (2004). "Modeling Regional Interdependencies Using a Global Error-Correcting Macroeconometric Model." *JBES*, 22(2), 129-162.
- Dees, S., di Mauro, F., Pesaran, M.H. & Smith, L.V. (2007). "Exploring the International Linkages of the Euro Area: A Global VAR Analysis." *Journal of Applied Econometrics*, 22(1), 1-38.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

from chronobox import GVAR, VAR, SVAR, HistoricalDecomposition

sys.path.insert(0, os.path.join("..", "utils"))
from plot_helpers import plot_spillover_network, plot_historical_decomposition

%matplotlib inline
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.facecolor"] = "white"
np.set_printoptions(precision=4, suppress=True)

print("All imports loaded successfully.")

## 1. Loading Multi-Country Panel Data

Our dataset contains quarterly observations for 5 countries: US, UK, Germany (DE), Japan (JP), and Brazil (BR). Each country has 4 macroeconomic variables: GDP growth, inflation, interest rate, and unemployment.

The data is generated with a **known trade-weight structure** where countries interact through bilateral spillovers.

In [ ]:
# Load multi-country panel data
data_path = os.path.join("..", "data", "us_macro_panel.csv")
df_panel = pd.read_csv(data_path, parse_dates=["date"])

print(f"Panel shape: {df_panel.shape}")
print(f"Countries: {df_panel['country'].unique().tolist()}")
print(f"Variables: {[c for c in df_panel.columns if c not in ['date', 'country']]}")
print(f"Period: {df_panel['date'].min()} to {df_panel['date'].max()}")
print(f"Observations per country: {df_panel.groupby('country').size().values[0]}")

# Show first few rows
df_panel.head(10)

In [ ]:
# Visualize GDP across countries
countries = df_panel["country"].unique().tolist()
var_names = ["gdp", "inflation", "interest_rate", "unemployment"]
colors_country = {"US": "steelblue", "UK": "firebrick", "DE": "darkgreen", 
                  "JP": "darkorange", "BR": "purple"}

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
titles = ["GDP Growth", "Inflation", "Interest Rate", "Unemployment"]

for idx, (ax, var, title) in enumerate(zip(axes.flat, var_names, titles)):
    for country in countries:
        mask = df_panel["country"] == country
        dates = df_panel.loc[mask, "date"]
        values = df_panel.loc[mask, var]
        ax.plot(dates, values, label=country, color=colors_country[country], linewidth=1.2)
    ax.set_title(title, fontsize=11)
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=8)

plt.suptitle("Multi-Country Macroeconomic Data", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "gvar_panel_data.png"), bbox_inches="tight")
plt.show()

## 2. Trade Weight Matrix

The **trade weight matrix** $W$ is fundamental to the GVAR. Entry $w_{ij}$ represents the importance of country $j$ as a trading partner for country $i$. Properties:

- Diagonal is zero: $w_{ii} = 0$ (no self-weight)
- Row-stochastic: $\sum_j w_{ij} = 1$ (weights sum to 1 for each country)

These weights determine how foreign variables are constructed:
$$y^*_{it} = \sum_{j \neq i} w_{ij} y_{jt}$$

In [ ]:
# Define trade weight matrix (stylized bilateral trade shares)
# Based roughly on actual trade patterns
np.random.seed(42)
n_countries = 5

# Stylized trade weights reflecting real-world patterns
W = np.array([
    [0.00, 0.20, 0.25, 0.20, 0.35],  # US: trades with all, strong with BR
    [0.30, 0.00, 0.35, 0.15, 0.20],  # UK: strong with US and DE (EU)
    [0.25, 0.30, 0.00, 0.15, 0.30],  # DE: strong with UK and BR
    [0.35, 0.15, 0.15, 0.00, 0.35],  # JP: strong with US and BR
    [0.30, 0.15, 0.20, 0.35, 0.00],  # BR: strong with US and JP
])

# Normalize rows to sum to 1
W = W / W.sum(axis=1, keepdims=True)

print("Trade weight matrix W (row-stochastic):")
print(pd.DataFrame(W.round(3), index=countries, columns=countries))
print(f"\nRow sums (should be 1): {W.sum(axis=1).round(4)}")

# Visualize trade weights as heatmap
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(W, cmap="YlOrRd", aspect="equal")
ax.set_xticks(range(n_countries))
ax.set_xticklabels(countries)
ax.set_yticks(range(n_countries))
ax.set_yticklabels(countries)
ax.set_xlabel("To (trading partner)")
ax.set_ylabel("From (home country)")
ax.set_title("Bilateral Trade Weight Matrix", fontsize=13)

for i in range(n_countries):
    for j in range(n_countries):
        ax.text(j, i, f"{W[i,j]:.2f}", ha="center", va="center", fontsize=10)

fig.colorbar(im, ax=ax, shrink=0.8, label="Weight")
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "gvar_trade_weights.png"), bbox_inches="tight")
plt.show()

## 3. Fitting the GVAR Model

The GVAR estimation proceeds in two stages:

1. **Country-level VARX**: for each country $i$, estimate a VARX model with domestic variables and trade-weighted foreign variables
2. **Global solution**: stack all country models and solve for the global companion form

The `chronobox.GVAR` class handles both stages automatically.

In [ ]:
# Prepare data dictionary: country_name -> ndarray (T, k_vars)
data_dict = {}
for country in countries:
    mask = df_panel["country"] == country
    data_dict[country] = df_panel.loc[mask, var_names].values

print("Data dictionary:")
for country, arr in data_dict.items():
    print(f"  {country}: shape {arr.shape}")

# Fit GVAR
gvar_model = GVAR(trade_weights=W, domestic_lags=1, foreign_lags=1)
gvar_results = gvar_model.fit(data_dict)

print(f"\nGVAR estimation complete:")
print(f"  Countries: {gvar_results.n_countries}")
print(f"  Total variables: {gvar_results.k_total}")
print(f"  Observations: {gvar_results.n_obs}")
print(f"  Domestic lags: {gvar_results.domestic_lags}")
print(f"  Foreign lags: {gvar_results.foreign_lags}")
print(f"  System stable: {gvar_results.is_stable}")
print(f"  Max eigenvalue: {np.max(np.abs(gvar_results.eigenvalues)):.4f}")

In [ ]:
# Examine country-specific results
for country in countries:
    cr = gvar_results.country_results[country]
    print(f"\n{country}:")
    print(f"  Domestic coef (Phi): {cr['Phi'][0].shape}")
    print(f"  Foreign coef (Lambda_0): {cr['Lambda'][0].shape}")
    print(f"  Residual std: {np.sqrt(np.diag(cr['Sigma'])).round(4)}")

# Stability analysis: eigenvalue plot
fig, ax = plt.subplots(figsize=(8, 8))
eigs = gvar_results.eigenvalues
theta = np.linspace(0, 2 * np.pi, 100)
ax.plot(np.cos(theta), np.sin(theta), "k-", linewidth=0.5, alpha=0.5)
ax.scatter(eigs.real, eigs.imag, color="firebrick", s=30, zorder=5)
ax.set_xlabel("Real")
ax.set_ylabel("Imaginary")
ax.set_title(f"GVAR Eigenvalues (max |λ| = {np.max(np.abs(eigs)):.4f})", fontsize=13)
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Generalized Impulse Response Functions (GIRF)

Unlike Cholesky IRFs, **Generalized IRFs** (Pesaran & Shin, 1998) do not depend on variable ordering. The GIRF for a shock to variable $j$ is:

$$\text{GIRF}(h) = \Phi_h \Sigma e_j / \sqrt{\sigma_{jj}}$$

where $\Phi_h$ are the MA coefficients of the global system, $\Sigma$ is the global residual covariance, and $\sigma_{jj}$ is the variance of the shocked variable.

This allows us to trace how a shock in **one country** propagates to **all other countries** through the trade linkages.

In [ ]:
# GIRF: Shock to US GDP and trace propagation to all countries
girf_us_gdp = gvar_results.girf(shock_country="US", shock_var=0, periods=20)
print(f"GIRF shape: {girf_us_gdp.shape}  (horizons, k_total={gvar_results.k_total})")

# Extract country-specific responses to US GDP shock
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
horizons = np.arange(girf_us_gdp.shape[0])

for idx, country in enumerate(countries):
    ax = axes.flat[idx]
    
    # Get response for this country
    irf_country = gvar_results.irf_country(
        shock_country="US", shock_var=0, response_country=country, periods=20
    )
    
    for v, (var_name, style) in enumerate(zip(var_names, ["-", "--", "-.", ":"])):
        ax.plot(horizons, irf_country[:, v], linestyle=style, linewidth=1.5, label=var_name)
    
    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_title(f"{country}", fontsize=12)
    ax.set_xlabel("Quarters")
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=7)

# Remove extra subplot
axes.flat[-1].set_visible(False)

plt.suptitle("GIRF: Response to a US GDP Shock (1 std. dev.)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "gvar_girf_us_gdp.png"), bbox_inches="tight")
plt.show()

## 5. Spillover Analysis: Cross-Country GDP Transmission

A key application of GVAR is quantifying **spillovers** between countries. We compute the peak GIRF response of each country's GDP to a GDP shock in every other country, constructing a **spillover matrix**.

In [ ]:
# Build GDP spillover matrix: peak response of country i's GDP to country j's GDP shock
spillover_matrix = np.zeros((n_countries, n_countries))

for j, shock_country in enumerate(countries):
    for i, resp_country in enumerate(countries):
        irf_ij = gvar_results.irf_country(
            shock_country=shock_country, shock_var=0,  # GDP shock
            response_country=resp_country, periods=20
        )
        # Peak absolute response of GDP (variable 0)
        spillover_matrix[i, j] = np.max(np.abs(irf_ij[:, 0]))

print("GDP Spillover Matrix (peak |GIRF| of GDP):")
print(pd.DataFrame(spillover_matrix.round(4), index=countries, columns=countries))
print(f"\nTotal spillover received (row sum excl. diagonal):")
for i, c in enumerate(countries):
    off_diag = spillover_matrix[i, :].sum() - spillover_matrix[i, i]
    print(f"  {c}: {off_diag:.4f}")

# Visualize using the plot helper
fig = plot_spillover_network(
    spillover_matrix * 100,  # scale for readability
    variable_names=countries,
    threshold=0.1,
    title="GDP Spillover Network (Peak |GIRF| × 100)",
)
plt.savefig(os.path.join("..", "outputs", "gvar_spillover_network.png"), bbox_inches="tight")
plt.show()

## Part II: Historical Decomposition

**Historical decomposition** answers the question: "How much of the observed movement in a variable is attributable to each structural shock?"

Given an SVAR with structural shocks $\varepsilon_t$ and structural MA coefficients $\Theta_h = \Phi_h A_0^{-1}$:

$$Y_t = \text{base}_t + \sum_{k=1}^{K} \text{HD}_{k,t}$$

where:
$$\text{HD}_{k,t} = \sum_{j=0}^{t} \Theta_j[:,k] \cdot \varepsilon_{t-j,k}$$

is the cumulative contribution of shock $k$ up to time $t$.

**References:**
- Kilian, L. & Lutkepohl, H. (2017). *Structural Vector Autoregressive Analysis*. Cambridge University Press.
- Burbidge, J. & Harrison, A. (1985). "An Historical Decomposition of the Great Depression." *Journal of Monetary Economics*, 16(1), 45-54.

In [ ]:
# Load US macro data for historical decomposition
data_path = os.path.join("..", "data", "us_macro_quarterly.csv")
df_us = pd.read_csv(data_path, parse_dates=["date"])
df_us.set_index("date", inplace=True)

us_vars = ["gdp", "inflation", "fed_funds", "unemployment"]
endog = df_us[us_vars].values

# Step 1: Fit reduced-form VAR
var_model = VAR(lags=4, trend="c")
var_results = var_model.fit(endog)
print(f"VAR(4) fitted: {var_results.n_obs} obs, {var_results.k_vars} vars")

# Step 2: Identify structural shocks via Cholesky
svar = SVAR(var_results, method="cholesky")
svar_results = svar.fit()
print(f"SVAR identified: method={svar_results.method}")
print(f"Structural shocks shape: {svar_results.structural_shocks.shape}")

In [ ]:
# Step 3: Compute historical decomposition
shock_names = ["GDP shock", "Inflation shock", "Monetary shock", "Unemp. shock"]
hd = HistoricalDecomposition(
    svar_results,
    shock_names=shock_names,
    variable_names=us_vars,
)

hd_result = hd.result
print(f"Historical decomposition computed:")
print(f"  Decomposition shape: {hd_result.decomposition.shape}  (T, K_shocks, K_vars)")
print(f"  Base shape: {hd_result.base.shape}")
print(f"  Observations: {hd_result.n_obs}")

# Verify decomposition: base + sum(HD) = observed
print(f"\nDecomposition verification: {hd_result.verify_decomposition()}")

In [ ]:
# Plot historical decomposition for GDP
dates_hd = df_us.index[4:]  # skip first p=4 lags

fig = plot_historical_decomposition(
    hd_result.decomposition,
    shock_names=shock_names,
    dates=dates_hd[:hd_result.n_obs],
    target_var=0,  # GDP
    title="Historical Decomposition: GDP Growth",
)
plt.savefig(os.path.join("..", "outputs", "gvar_hd_gdp.png"), bbox_inches="tight")
plt.show()

In [ ]:
# Plot historical decomposition for all 4 variables
fig, axes = plt.subplots(4, 1, figsize=(16, 16), sharex=True)
var_titles = ["GDP Growth", "Inflation", "Fed Funds Rate", "Unemployment"]
cmap = plt.cm.Set2

for var_idx, (ax, var_title) in enumerate(zip(axes, var_titles)):
    decomp = hd_result.decomposition[:, :, var_idx]  # (T, K_shocks) for this variable
    
    pos = np.maximum(decomp, 0)
    neg = np.minimum(decomp, 0)
    bottom_pos = hd_result.base[:, var_idx].copy()
    bottom_neg = hd_result.base[:, var_idx].copy()
    
    t_axis = np.arange(hd_result.n_obs)
    
    for k in range(4):
        color = cmap(k / 3)
        ax.bar(t_axis, pos[:, k], bottom=bottom_pos, color=color, 
               alpha=0.8, width=1.0, label=shock_names[k] if var_idx == 0 else "")
        ax.bar(t_axis, neg[:, k], bottom=bottom_neg, color=color, alpha=0.8, width=1.0)
        bottom_pos += pos[:, k]
        bottom_neg += neg[:, k]
    
    # Observed series
    ax.plot(t_axis, hd_result.observed[:, var_idx], "k-", linewidth=1.5, 
            label="Observed" if var_idx == 0 else "")
    
    ax.set_ylabel(var_title, fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color="k", linewidth=0.5)

axes[0].legend(fontsize=8, loc="upper right", ncol=5)
plt.suptitle("Historical Decomposition: All Variables", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "gvar_hd_all_variables.png"), bbox_inches="tight")
plt.show()

In [ ]:
# Analyze contribution of each shock using DataFrame interface
hd_gdp_df = hd_result.to_dataframe(variable=0)
print("Historical Decomposition DataFrame (GDP):")
print(hd_gdp_df.describe().round(3))

# Which shock explains the most GDP variance?
print("\n\nVariance contribution of each shock to GDP:")
for k, name in enumerate(shock_names):
    var_contrib = np.var(hd_result.decomposition[:, k, 0])
    total_var = np.var(hd_result.observed[:, 0])
    print(f"  {name}: {var_contrib/total_var*100:.1f}%")

## Exercicio

### Exercicio 1: Spillovers entre paises via GVAR

Analise como um choque de politica monetaria (interest_rate) nos EUA se propaga para os outros paises.

1. Compute o GIRF para um choque na `interest_rate` dos EUA (variavel index 2)
2. Para cada pais, extraia a resposta do GDP e da inflacao
3. Qual pais e mais afetado pelo choque monetario dos EUA?
4. A resposta e consistente com os pesos comerciais? Paises com maior peso bilateral com os EUA devem ser mais afetados.

In [ ]:
# --- Exercicio 1: Seu codigo aqui ---

# Dica:
# girf_us_rate = gvar_results.girf(shock_country="US", shock_var=2, periods=20)
# for country in countries:
#     irf_c = gvar_results.irf_country("US", 2, country, periods=20)
#     peak_gdp = np.max(np.abs(irf_c[:, 0]))
#     print(f"{country}: peak GDP response = {peak_gdp:.4f}")
# 
# # Compare with trade weights from US column
# us_col = W[:, 0]  # weights that other countries assign to US
# print("Trade weights TO US:", dict(zip(countries, us_col.round(3))))

### Exercicio 2: Historical decomposition com identificacao alternativa

A historical decomposition depende da identificacao estrutural. Compare:

1. Historical decomposition com Cholesky e ordering `[gdp, inflation, fed_funds, unemployment]` (atual)
2. Historical decomposition com ordering alternativo `[inflation, gdp, fed_funds, unemployment]`
3. Os resultados sao sensiveis ao ordenamento? Em quais periodos a sensibilidade e maior?
4. Qual choque e o principal driver do GDP em ambos os casos?

In [ ]:
# --- Exercicio 2: Seu codigo aqui ---

# Dica:
# alt_vars = ["inflation", "gdp", "fed_funds", "unemployment"]
# endog_alt = df_us[alt_vars].values
# var_alt = VAR(lags=4, trend="c")
# var_res_alt = var_alt.fit(endog_alt)
# svar_alt = SVAR(var_res_alt, method="cholesky")
# svar_res_alt = svar_alt.fit()
# hd_alt = HistoricalDecomposition(svar_res_alt, shock_names=alt_vars)
# 
# # Compare variance contribution of each shock to GDP (index 1 in alt ordering)
# for k in range(4):
#     var_contrib = np.var(hd_alt.decomposition[:, k, 1])
#     print(f"{alt_vars[k]}: {var_contrib:.4f}")

---

## Resumo

Neste notebook aprendemos:

1. **GVAR**: framework multi-pais que conecta modelos VARX nacionais via matrizes de peso comercial, permitindo estudar interdependencias globais

2. **Matrizes de peso**: a estrutura de comercio bilateral determina como choques se propagam entre paises — pesos maiores implicam transmissao mais forte

3. **GIRF**: funcoes de resposta ao impulso generalizadas que nao dependem do ordenamento das variaveis, ideais para sistemas multi-pais

4. **Spillover analysis**: quantificacao da transmissao de choques entre paises, com visualizacao via redes/heatmaps

5. **Historical decomposition**: decomposicao dos movimentos observados nas contribuicoes de cada choque estrutural identificado via SVAR

6. **Verificacao**: `verify_decomposition()` confirma que base + soma(HD) = observado

## Referencias

- Pesaran, M.H., Schuermann, T. & Weiner, S.M. (2004). "Modeling Regional Interdependencies Using a Global Error-Correcting Macroeconometric Model." *JBES*, 22(2), 129-162.
- Dees, S., di Mauro, F., Pesaran, M.H. & Smith, L.V. (2007). "Exploring the International Linkages of the Euro Area." *JAE*, 22(1), 1-38.
- Kilian, L. & Lutkepohl, H. (2017). *Structural Vector Autoregressive Analysis*. Cambridge University Press.
- Burbidge, J. & Harrison, A. (1985). "An Historical Decomposition of the Great Depression." *JME*, 16(1), 45-54.